<a href="https://colab.research.google.com/github/salonapaliwal7/Neural_Networks/blob/main/Neural_network_TF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import tensorflow as tf
import numpy as np
import time

In [ ]:
# import tensorflow as tf

# x = tf.Variable(2.0)  # x = 2

# with tf.GradientTape() as tape2:
#     with tf.GradientTape() as tape1:
#         y = x ** 3     # y = x^3

#     # First derivative dy/dx
#     dy_dx = tape1.gradient(y, x) #3x^2

# # Second derivative d²y/dx²
# d2y_dx2 = tape2.gradient(dy_dx, x)#6x

In [5]:
# for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

N = 500 # samples
# Generate random numbers from a uniform distribution between -1 and 1.
X = np.random.uniform(-1,1,size=(N,1)).astype(np.float32) # shape = (N,1)

In [6]:
# y= 3x+2+small noise
true_w = 3
true_b = 2
noise = np.random.normal(0.0,0.1,size=(N,1)).astype(np.float32) # will generate 500 noise value between 0.0 to 0.1
noise.shape

(500, 1)

In [21]:
Y = (true_w*X + true_b + noise).astype(np.float32)

# convert to tf tensors
X_tf = tf.constant(X) # inputs
Y_tf = tf.constant(Y) # targets

In [22]:
# input = 1, output = 1
input_dim = 1
hidden_layers = 8
output_dim = 1


# Initialise weights and bias
W1 = tf.Variable(tf.random.normal([input_dim,hidden_layers],stddev=0.5),name="W1")
b1 = tf.Variable(tf.zeros([hidden_layers]),name = "b1")

W2 = tf.Variable(tf.random.normal([hidden_layers,output_dim],stddev=0.5),name="W2")
b2 = tf.Variable(tf.zeros([output_dim]),name = "b2")

In [23]:
W1

<tf.Variable 'W1:0' shape=(1, 8) dtype=float32, numpy=
array([[ 0.00462325, -0.33103138, -0.37051344,  0.5992631 ,  0.4318104 ,
         0.19628781,  0.46728748, -0.08008733]], dtype=float32)>

In [24]:
b1

<tf.Variable 'b1:0' shape=(8,) dtype=float32, numpy=array([0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)>

In [25]:
W2

<tf.Variable 'W2:0' shape=(8, 1) dtype=float32, numpy=
array([[ 0.0153802 ],
       [ 0.14508986],
       [ 0.64148873],
       [ 0.40635768],
       [-0.39724588],
       [ 0.10011418],
       [ 0.901625  ],
       [-0.51882464]], dtype=float32)>

In [26]:
b2

<tf.Variable 'b2:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>

In [30]:
def forward_pass(X):
  z = tf.matmul(X, W1) + b1           # shape(input, hidden_layer)
  a = tf.nn.relu(z)                  # activation
  output = tf.matmul(a,W2) + b2      # shape(hidden_layer, ouput)
  return output


# loss - Mean Squared Error
def mse_loss(y_pred,y_true):
  return tf.reduce_mean(tf.square(y_pred, y_true))


In [31]:
learning_rate = 0.1
epochs = 200
batch_size = 32
steps_per_epoch = int(np.ceil(N/batch_size))


In [33]:
start_time = time.time()
for epoch in range(1, epochs + 1):
    idx = np.random.permutation(N) # randomly gives 500 index
    X_shuffled = X_tf.numpy()[idx]
    Y_shuffled = Y_tf.numpy()[idx]

    epoch_loss = 0.0

    for step in range(steps_per_epoch):
        start = step * batch_size
        end = min(start + batch_size, N)
        x_batch = tf.constant(X_shuffled[start:end])
        y_batch = tf.constant(Y_shuffled[start:end])

        # Record operations for automatic differentiation
        with tf.GradientTape() as tape:
            y_pred = forward_pass(x_batch)          # forward
            loss = mse_loss(y_pred, y_batch)        # compute loss

        # Backward: compute gradients of loss w.r.t. all trainable variables
        grads = tape.gradient(loss, [W1, b1, W2, b2])

        # Manual SGD parameter update: param = param - lr * grad
        #w1 =w1 - lr*grad
        W1.assign_sub(learning_rate * grads[0])
        b1.assign_sub(learning_rate * grads[1])
        W2.assign_sub(learning_rate * grads[2])
        b2.assign_sub(learning_rate * grads[3])

        epoch_loss += loss.numpy() * (end - start)

        # average loss over dataset
    epoch_loss /= N

    # print some progress
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{epochs} - Loss: {epoch_loss:.6f}")

end_time = time.time()
print(f"Training finished in {end_time - start_time:.3f} seconds")

y_pred_final = forward_pass(X_tf)
final_loss = mse_loss(y_pred_final, Y_tf).numpy()
print("\nFinal MSE on training set:", final_loss)

Epoch   1/200 - Loss: 0.000001
Epoch  20/200 - Loss: 0.000001
Epoch  40/200 - Loss: 0.000001
Epoch  60/200 - Loss: 0.000001
Epoch  80/200 - Loss: 0.000000
Epoch 100/200 - Loss: 0.000000
Epoch 120/200 - Loss: 0.000000
Epoch 140/200 - Loss: 0.000000
Epoch 160/200 - Loss: 0.000000
Epoch 180/200 - Loss: 0.000000
Epoch 200/200 - Loss: 0.000000
Training finished in 39.369 seconds

Final MSE on training set: 3.806198e-07
